# ACLED aggregated data — Milestone 3 preprocessing

Merges the six regional `.xlsx` files in `ACLED Data/`, cleans and types columns, maps country labels toward World Bank `country` strings, and writes `output/` artifacts (see `acled_pipeline.py`).

**BCNF / 3NF:** Loads are split into **`acled_country`** (`country` → `country_wdi`), **`acled_source_area`** (`source_area_id` → geography), and **`acled_weekly_events`** (full event grain + measures). Optional **`acled_denormalized_joined.*`** repeats the join for analysis only.

**Fact primary key:** `(week, source_area_id, event_type, sub_event_type)`. See `load_acled.sql` for `CREATE TABLE` and `COPY` order.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Run this notebook with the working directory set to the project root (folder that contains `ACLED Data/`).
ROOT = Path.cwd()
if not (ROOT / "ACLED Data").is_dir():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "acled_preprocessing"))

from acled_pipeline import clean_dataframe, export_artifacts, load_and_merge

assert (ROOT / "ACLED Data").is_dir(), "Set notebook cwd to CIS 5500/Project"
ACLED_DIR = ROOT / "ACLED Data"
OUT_DIR = ROOT / "acled_preprocessing" / "output"

ROOT, ACLED_DIR

## 1. Load, merge, and assert columns

Each file is tagged with `source_file` and `ddl_table_key` (six tables from Milestone 2).

In [ ]:
raw = load_and_merge(ACLED_DIR)
raw.groupby("source_file").size().rename("rows").to_frame()

## 2. Clean, validate, and map countries

- Drops rows with missing `source_area_id` (65) or `admin1` (8).
- Keeps `population_exposure` nulls (wide-area coverage).
- Drops rows with invalid lat/lon (7).
- Adds `country_wdi` using `ACLED_TO_WDI_COUNTRY` (extend `acled_pipeline.py` as needed).

In [ ]:
clean, stats = clean_dataframe(raw)
pd.Series(stats).to_frame("count")

In [ ]:
clean.head()

## 3. Export BCNF tables, optional denormalized join, regional fact slices

Writes `acled_country.csv`, `acled_source_area.csv`, `acled_weekly_events.csv`, `acled_denormalized_joined.csv`, `by_ddl_table/*.csv`, and `preprocessing_summary.json`.

In [ ]:
export_artifacts(clean, OUT_DIR, stats)
sorted(p.name for p in OUT_DIR.glob("acled_*.csv")), sorted(p.name for p in (OUT_DIR / "by_ddl_table").glob("*.csv"))